# Multimodal Human Activity Recognition on MMAct (Kaggle GPU Pipeline)

This notebook executes the software-only Multimodal Human Activity Recognition (HAR) pipeline on the **MMAct dataset** pairing video clips and wearable inertial sensors (phone accelerometer, watch accelerometer, gyroscope) across 10 verified activity classes:
`crouching`, `jumping`, `kicking`, `pointing`, `running`, `sitting`, `standing`, `using_phone`, `walking`, `waving_hand`.

---
### Kaggle Setup Instructions:
1. **Enable GPU Accelerator:**
   - In the right-hand panel under **Session Options** -> **Accelerator**, select **GPU T4 x2** or **GPU P100**.
2. **Enable Internet Access:**
   - In the right-hand panel under **Session Options** -> **Internet**, toggle to **On** (required for `git clone` and installing dependencies).
3. **Attach MMAct Dataset:**
   - Click **+ Add Input** in the top right.
   - If you uploaded your MMAct files to Kaggle Datasets (e.g., zip archives or pre-extracted folder), attach that dataset.
   - The notebook automatically scans `/kaggle/input/` to detect your dataset folder or unpack ZIP archives.

In [ ]:
# Step 1: GPU & PyTorch Environment Verification
!nvidia-smi

import torch
print('=' * 60)
print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name :', torch.cuda.get_device_name(0))
    print('Device Count    :', torch.cuda.device_count())
    print('Device Capability:', torch.cuda.get_device_capability(0))
else:
    print('[WARNING] CUDA GPU is not detected! Ensure GPU Accelerator is enabled in Kaggle Settings.')
print('=' * 60)


In [ ]:
# Step 2: Clone Repository & Install Dependencies
import os
from pathlib import Path

%cd /kaggle/working

REPO_URL = 'https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git'
REPO_DIR = '/kaggle/working/HAR-MMAct'

if not os.path.exists(REPO_DIR):
    print(f'Cloning repository from {REPO_URL}...')
    !git clone {REPO_URL}
else:
    print(f'Repository already present at {REPO_DIR}. Pulling latest changes...')
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

# Install required dependencies
!pip install -q -r requirements.txt
print('Dependencies installed successfully.')


In [ ]:
# Step 3: Discover & Prepare Genuine MMAct Dataset from Kaggle Input
import glob
import shutil
import zipfile
from pathlib import Path

kaggle_input = Path('/kaggle/input')
project_root = Path('/kaggle/working/HAR-MMAct')
target_data_dir = project_root / 'data' / 'genuine_mmact'
target_data_dir.mkdir(parents=True, exist_ok=True)

print('Scanning /kaggle/input for MMAct dataset files...')
input_dirs = list(kaggle_input.glob('*'))
print('Found input directories:', [d.name for d in input_dirs])

# Check if dataset is already extracted in Kaggle input
found_extracted = False
for d in input_dirs:
    # Look for trimmed or genuine_mmact structure
    candidate_trimmed = d / 'trimmed' if (d / 'trimmed').exists() else (d / 'genuine_mmact' / 'trimmed' if (d / 'genuine_mmact' / 'trimmed').exists() else d)
    if (candidate_trimmed / 'video').exists() and (candidate_trimmed / 'sensors').exists():
        print(f'Found extracted MMAct dataset at: {candidate_trimmed}')
        dest_trimmed = target_data_dir / 'trimmed'
        if not dest_trimmed.exists():
            try:
                os.symlink(candidate_trimmed, dest_trimmed)
                print(f'Symlinked {candidate_trimmed} -> {dest_trimmed}')
            except Exception as e:
                shutil.copytree(candidate_trimmed, dest_trimmed, dirs_exist_ok=True)
                print(f'Copied {candidate_trimmed} -> {dest_trimmed}')
        found_extracted = True
        break

# If not pre-extracted, look for ZIP archives (Option 2 dataset)
if not found_extracted:
    zip_files = list(kaggle_input.rglob('*.zip'))
    print(f'Found {len(zip_files)} zip archives in /kaggle/input:')
    for z in zip_files:
        print(' -', z.name)
    
    dest_downloads = project_root / 'data' / 'downloads'
    dest_downloads.mkdir(parents=True, exist_ok=True)
    for z in zip_files:
        tgt = dest_downloads / z.name
        if not tgt.exists():
            try:
                os.symlink(z, tgt)
            except Exception:
                shutil.copy(z, tgt)
    
    # Run automatic ingestion
    from src.data.ingestion import DatasetIngestionPipeline
    ingestion = DatasetIngestionPipeline(
        incoming_dir=dest_downloads,
        extracted_dir=target_data_dir / 'trimmed'
    )
    manifest = ingestion.scan_and_ingest()
    print('Ingestion complete. Manifest status:', manifest.get('status'))

# Verify indexing of genuine MMAct data
from src.data.indexer import MMActIndexer
indexer = MMActIndexer(dataset_root=target_data_dir)
report = indexer.index()
print('\n' + '=' * 60)
print('MMACT INDEXER VERIFICATION:')
print(f'  - Paired Samples : {report.paired_samples_count}')
print(f'  - Videos Scanned : {report.total_video_files_found}')
print(f'  - Sensors Scanned: {report.total_sensor_files_found}')
print('=' * 60)


In [ ]:
# Step 4: Run Exact 3-Epoch Benchmark on Genuine MMAct Data
# Split strategy: subject-independent (subject8 -> train, subject16 -> val & test)
!python scripts/train.py \
    --data_root data/genuine_mmact \
    --epochs 3 \
    --batch_size 8 \
    --split_strategy subject \
    --mode multimodal \
    --video_backbone resnet18 \
    --sensor_backbone inception_time \
    --output_dir outputs \
    --cache_dir outputs/cache


In [ ]:
# Step 5: Standalone Evaluation & Unimodal Baseline Comparison
# Evaluates best_model.pth on held-out subject16 test split across:
# 1. Multimodal (Video + Sensors)
# 2. Video-Only Baseline
# 3. Sensor-Only Baseline
!python scripts/evaluate.py \
    --checkpoint outputs/checkpoints/best_model.pth \
    --data_root data/genuine_mmact \
    --output_file outputs/metrics.json \
    --plots_dir outputs/plots


In [ ]:
# Step 6: Display Generated Benchmark Plots Inline
from IPython.display import Image, display
import os

plots_to_show = [
    ('Training Loss & Accuracy Trajectories', 'outputs/plots/training_curves.png'),
    ('Train vs Validation Loss (Standalone)', 'outputs/plots/train_vs_val_loss.png'),
    ('Train vs Validation Accuracy (Standalone)', 'outputs/plots/train_vs_val_accuracy.png'),
    ('Validation Confusion Matrix', 'outputs/plots/val_confusion_matrix.png'),
    ('Test Confusion Matrix (Subject-Independent)', 'outputs/plots/test_confusion_matrix.png'),
    ('One-vs-Rest ROC Curves', 'outputs/plots/test_roc_ovr.png'),
    ('Precision-Recall Curves', 'outputs/plots/test_pr_curves.png'),
]

for title, p in plots_to_show:
    if os.path.exists(p):
        print('=' * 60)
        print(title)
        print('=' * 60)
        display(Image(filename=p))


In [ ]:
# Step 7: Sample Interactive Multimodal Inference
!python scripts/infer.py \
    --video data/genuine_mmact/trimmed/video/cross_scene/trainval/subject16/cam1/scene1/session1/crouching.mp4 \
    --sensor_phone data/genuine_mmact/trimmed/sensors/acc_phone_clip/subject16/scene1/session1/crouching.csv \
    --sensor_watch data/genuine_mmact/trimmed/sensors/acc_watch_clip/subject16/scene1/session1/crouching.csv \
    --sensor_gyro data/genuine_mmact/trimmed/sensors/gyro_clip/subject16/scene1/session1/crouching.csv \
    --checkpoint outputs/checkpoints/best_model.pth


In [ ]:
# Step 8: Package Artifacts into /kaggle/working/ for Easy Download
import shutil
import json
from pathlib import Path

# Print training summary
summary_path = Path('outputs/training_summary.json')
if summary_path.exists():
    with open(summary_path) as f:
        print('Training Summary:')
        print(json.dumps(json.load(f), indent=2))

# Copy outputs to /kaggle/working/outputs so Kaggle exposes them in the Output tab
dest_outputs = Path('/kaggle/working/outputs')
if Path('outputs').resolve() != dest_outputs.resolve():
    shutil.copytree('outputs', dest_outputs, dirs_exist_ok=True)

# Create a single zip archive for 1-click download
shutil.make_archive('/kaggle/working/mmact_benchmark_results', 'zip', 'outputs')
print('Results zipped successfully to: /kaggle/working/mmact_benchmark_results.zip')
print('You can download this zip and all output files directly from the Kaggle Output viewer.')
